# 01 — Dataset Preparation

Download/sample the GenImage subset, standardize images, and build the dataset that every later notebook reads from. See data/README.md and configs/experiments.yaml for more details.

In [1]:
import sys
from pathlib import Path
from typing import TypedDict
import random
import pandas as pd
from sklearn.model_selection import train_test_split
from PIL import Image

REPO_ROOT = Path.cwd().parent  # notebooks
sys.path.insert(0, str(REPO_ROOT))

from src.data.standardize import standardize_image, CorruptImageError
from src.data.config import load_config, active_sample_size, raw_generator_dir

class SampledImage(TypedDict, total=False):
    """Metadata for an image sample used during dataset preparation.

    The record stores the original image path, its label, generator,
    and the path to the standardized image after preprocessing.
    """

    src_path: Path  # original file location under paths.raw_data_root
    label: str  # "ai" | "real"
    generator: str  # generator name for AI images, "real" for real images
    dst_path: Path  # standardized JPEG location under data/processed/

## Config

Load the experiment settings from `configs/experiments.yaml`, including the random seed, selected generators, sample size, and preprocessing options.

In [4]:
config: dict = load_config()
SEED: int = config["seed"]
GENERATORS: list[str] = config["generators"]
N_PER_GENERATOR: int = active_sample_size(config)
PREP: dict = config["preprocessing"]

print(f"seed = {SEED}")
print(f"generators: {GENERATORS}")
print(f"active size: {config['sample_size']['active_size']} -> {N_PER_GENERATOR} images per generator")
print(f"preprocessing: {PREP}")

seed = 42
generators: ['biggan', 'glide', 'vqdm', 'sdv1_5', 'adm', 'midjourney', 'wukong']
active size: small -> 1000 images per generator
preprocessing: {'image_size': 224, 'color_mode': 'RGB', 'jpeg_quality': 90, 'strip_metadata': True}


## Locate the raw data

Set the path to the raw GenImage subset and check that the expected folder structure is available for each generator.

In [6]:
# The raw GenImage data is large (~35k images) and stored outside the repository and loaded from the configured path.
RAW_ROOT: Path = Path(config["paths"]["raw_data_root"])
assert RAW_ROOT.is_dir(), f"raw data root not found: {RAW_ROOT}"

print(f"Raw data root: {RAW_ROOT}\n")
for generator in GENERATORS:
    gen_dir: Path = raw_generator_dir(config, generator)
    # Only checking the "train" subfolder here as a quick existence probe
    ok: bool = gen_dir.is_dir() and (gen_dir / "train" / "ai").is_dir() and (gen_dir / "train" / "nature").is_dir()
    print(f"  {generator:<10} -> {gen_dir.name:<28} {'OK' if ok else 'MISSING'}")
    # Stop early if a generator is missing data.
    assert ok, f"missing/incomplete folder for generator {generator}: {gen_dir}"

Raw data root: C:\Users\valen\OneDrive\Desktop\Dataset\archive

  biggan     -> imagenet_ai_0419_biggan      OK
  glide      -> imagenet_glide               OK
  vqdm       -> imagenet_ai_0419_vqdm        OK
  sdv1_5     -> imagenet_ai_0424_sdv5        OK
  adm        -> imagenet_ai_0508_adm         OK
  midjourney -> imagenet_midjourney          OK
  wukong     -> imagenet_ai_0424_wukong      OK


## Sample images

Sample an equal number of AI and real images for each generator using the configured sample size and random seed.

In [ ]:
def list_raw_files(generator_dir: Path, class_dir: str) -> list[Path]:
    """Return all image files from the train and validation folders.

    Args:
        generator_dir: Root directory for the generator.
        class_dir: Image class, either "ai" or "nature".

    Returns:
        A sorted list of image file paths.
    """
    files: list[Path] = []
    for subset in ("train", "val"):
        d = generator_dir / subset / class_dir
        if d.is_dir():
            files += [p for p in d.iterdir() if p.is_file()]
    return sorted(files)

records: list[SampledImage] = []
for generator in GENERATORS:
    gen_dir = raw_generator_dir(config, generator)
    ai_pool: list[Path] = list_raw_files(gen_dir, "ai")
    real_pool: list[Path] = list_raw_files(gen_dir, "nature")
    assert len(ai_pool) >= N_PER_GENERATOR and len(real_pool) >= N_PER_GENERATOR, (
        f"{generator}: need {N_PER_GENERATOR} each, found {len(ai_pool)} ai / {len(real_pool)} real"
    )

    # Use the same seed for reproducible sampling.
    rng = random.Random(SEED)
    ai_sample: list[Path] = rng.sample(ai_pool, N_PER_GENERATOR)
    real_sample: list[Path] = rng.sample(real_pool, N_PER_GENERATOR)

    records += [{"src_path": p, "label": "ai", "generator": generator} for p in ai_sample]
    records += [{"src_path": p, "label": "real", "generator": "real"} for p in real_sample]

n_ai = sum(r["label"] == "ai" for r in records)
n_real = sum(r["label"] == "real" for r in records)
print(f"Sampled {len(records)} images total ({n_ai} ai, {n_real} real)")

Sampled 14000 images total (7000 ai, 7000 real)


## Standardize images

Apply the same preprocessing to all images by resizing them, converting them to RGB, removing metadata, and saving them with consistent JPEG compression (bias prevention).

In [8]:
PROCESSED_ROOT: Path = REPO_ROOT / "data" / "processed"


def processed_path(record: SampledImage) -> Path:
    """Build the destination path for a standardized image.

    Args:
        record: Metadata for the sampled image.

    Returns:
        Path where the standardized JPEG image will be saved.
    """
    new_name = f"{record['generator']}__{record['src_path'].stem}.jpg"
    return PROCESSED_ROOT / record["label"] / new_name


errors: list[str] = []
for i, record in enumerate(records):
    dst: Path = processed_path(record)
    record["dst_path"] = dst
    try:
        standardize_image(
            record["src_path"],
            dst,
            image_size=PREP["image_size"],
            color_mode=PREP["color_mode"],
            jpeg_quality=PREP["jpeg_quality"],
        )
    except CorruptImageError as e:
        # Keep track of files that could not be standardized.
        errors.append(str(e))
    if (i + 1) % 2000 == 0:
        print(f"  standardized {i + 1}/{len(records)}")

print(f"\nDone. {len(records) - len(errors)} standardized, {len(errors)} failed.")
for e in errors[:20]:
    print(" ", e)

# Drop any records that failed to standardize so the dataset only lists real files.
records = [r for r in records if r["dst_path"].exists()]

  standardized 2000/14000
  standardized 4000/14000
  standardized 6000/14000
  standardized 8000/14000
  standardized 10000/14000
  standardized 12000/14000
  standardized 14000/14000

Done. 14000 standardized, 0 failed.


## Build dataset

Create `data/dataset.csv` with columns: filename | label | generator | split | source_path.

In [9]:
# Keep the label and generator distribution consistent across splits.
stratification_labels: list[str] = [f"{r['label']}:{r['generator']}" for r in records]
idx: list[int] = list(range(len(records)))
ratios: dict[str, float] = config["standard_split"]

# Split into train and a temporary set containing validation and test samples.
train_idx, rest_idx, _, rest_stratification_labels = train_test_split(idx, stratification_labels, test_size=ratios["val"] + ratios["test"], stratify=stratification_labels, random_state=SEED)

# Split the remaining samples into validation and test sets.
val_frac_of_rest = ratios["val"] / (ratios["val"] + ratios["test"])
val_idx, test_idx = train_test_split(rest_idx, test_size=1 - val_frac_of_rest, stratify=rest_stratification_labels, random_state=SEED)

# Map each sample index to its assigned split.
split_of: dict[int, str] = {}
for i in train_idx:
    split_of[i] = "train"
for i in val_idx:
    split_of[i] = "val"
for i in test_idx:
    split_of[i] = "test"

# Build the dataset with paths, labels, generators, and splits.
rows: list[dict[str, str]] = [
    {
        "filename": r["dst_path"].relative_to(PROCESSED_ROOT).as_posix(),
        "label": r["label"],
        "generator": r["generator"],
        "split": split_of[i],
        "source_path": str(r["src_path"]),
    }
    for i, r in enumerate(records)
]

dataset: pd.DataFrame = pd.DataFrame(rows)
dataset_path: Path = REPO_ROOT / "data" / "dataset.csv"
dataset.to_csv(dataset_path, index=False)
print(f"Wrote {len(dataset)} rows to {dataset_path}")
dataset.head()

Wrote 14000 rows to c:\Users\valen\OneDrive\Desktop\robust-ai-image-detection\data\dataset.csv


,filename,label,generator,split,source_path
0,ai/biggan__243_biggan_00176.jpg,ai,biggan,val,C:\Users\valen\OneDrive\Desktop\Dataset\archiv...
1,ai/biggan__055_biggan_00136.jpg,ai,biggan,train,C:\Users\valen\OneDrive\Desktop\Dataset\archiv...
2,ai/biggan__557_biggan_00144.jpg,ai,biggan,train,C:\Users\valen\OneDrive\Desktop\Dataset\archiv...
3,ai/biggan__500_biggan_00122.jpg,ai,biggan,train,C:\Users\valen\OneDrive\Desktop\Dataset\archiv...
4,ai/biggan__463_biggan_00098.jpg,ai,biggan,test,C:\Users\valen\OneDrive\Desktop\Dataset\archiv...


## Sanity checks

Check the dataset size, label and generator distribution, file paths, and a few standardized images.

In [10]:
print("Split sizes:")
print(dataset["split"].value_counts(), "\n")

print("Label balance overall:")
print(dataset["label"].value_counts(), "\n")

print("AI counts per generator:")
print(dataset[dataset.label == "ai"].groupby("generator").size(), "\n")

print("Label balance per split:")
print(dataset.groupby(["split", "label"]).size().unstack(), "\n")

# No duplicate filenames in the dataset.
assert dataset["filename"].is_unique, "duplicate filenames in dataset!"

# Check that all referenced processed images exist.
missing: list[str] = [f for f in dataset["filename"] if not (PROCESSED_ROOT / f).is_file()]
assert not missing, f"{len(missing)} dataset rows point at missing files, e.g. {missing[:5]}"

# Check a few images for the expected size, color mode, and metadata.
sample_filenames: list[str] = random.Random(SEED).sample(list(dataset["filename"]), 5)
for f in sample_filenames:
    img = Image.open(PROCESSED_ROOT / f)
    assert img.size == (PREP["image_size"], PREP["image_size"]), f"{f}: bad size {img.size}"
    assert img.mode == PREP["color_mode"], f"{f}: bad mode {img.mode}"
    assert img.getexif() == {} or len(img.getexif()) == 0, f"{f}: EXIF metadata was not removed"

print("All sanity checks passed.")

Split sizes:
split
train    9800
val      2100
test     2100
Name: count, dtype: int64 

Label balance overall:
label
ai      7000
real    7000
Name: count, dtype: int64 

AI counts per generator:
generator
adm           1000
biggan        1000
glide         1000
midjourney    1000
sdv1_5        1000
vqdm          1000
wukong        1000
dtype: int64 

Label balance per split:
label    ai  real
split            
test   1050  1050
train  4900  4900
val    1050  1050 

All sanity checks passed.
